# Clase 4 — OOP II — OrderBook y PositionTracker

Construir el libro como objeto que contiene niveles, con métricas como métodos. Y un PositionTracker que consume objetos Fill. Aquí ves cómo los objetos se entrelazan.

**Hoy construyes:** clases OrderBook y PositionTracker (composición).

## Cómo usar este cuaderno

- **Núcleo (en clase):** ejercicios 1 a 3.
- **Si vamos bien:** ejercicios 4 en adelante.
- **Casa / auxiliares:** el cuaderno `*_auxiliary.ipynb`.

Inténtalo, ejecuta la comprobación (`assert`) y mira la solución solo si te atascas.

## 1. OrderBook con niveles

**Practicas:** atributos que son listas.

Define `OrderBook(bids, asks)` donde cada lado es una lista de tuplas `(price, size)`.

In [ ]:
class OrderBook:
    def __init__(self, bids, asks):
        pass

In [ ]:
b = OrderBook([(100,1)], [(101,2)])
assert b.bids == [(100,1)] and b.asks == [(101,2)]
print('ok')

### Solución guiada

```python
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = bids
        self.asks = asks
```

## 2. best_bid / best_ask / spread / mid

**Practicas:** métodos sobre estado.

Añade métodos `best_bid()`, `best_ask()`, `spread()`, `mid()`. (bids ordenados desc, asks asc; el mejor es el primero.)

In [ ]:
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x:-x[0])
        self.asks = sorted(asks, key=lambda x:x[0])
    # añade los métodos dentro de la clase de arriba
    pass

In [ ]:
b = OrderBook([(100,1),(99,1)], [(101,1),(102,1)])
assert b.best_bid()==100 and b.best_ask()==101
assert b.spread()==1 and b.mid()==100.5
print('ok')

### Solución guiada

```python
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x:-x[0])
        self.asks = sorted(asks, key=lambda x:x[0])
    def best_bid(self):
        return self.bids[0][0]
    def best_ask(self):
        return self.asks[0][0]
    def spread(self):
        return self.best_ask() - self.best_bid()
    def mid(self):
        return (self.best_bid() + self.best_ask()) / 2
```

## 3. Imbalance del nivel 1

**Practicas:** método con cálculo.

Añade `imbalance()` = (bid_size - ask_size)/(bid_size + ask_size) en el mejor nivel.

In [ ]:
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x:-x[0])
        self.asks = sorted(asks, key=lambda x:x[0])
    def imbalance(self):
        pass

In [ ]:
b = OrderBook([(100,3)], [(101,1)])
assert abs(b.imbalance() - 0.5) < 1e-9
print('ok')

### Solución guiada

```python
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x:-x[0])
        self.asks = sorted(asks, key=lambda x:x[0])
    def imbalance(self):
        bs = self.bids[0][1]; as_ = self.asks[0][1]
        return (bs - as_) / (bs + as_)
```

## 4. PositionTracker

**Practicas:** estado privado.

Define `PositionTracker` con `_cash=0`, `_position=0` y `apply_fill(fill)` que sume `fill.cash_flow()` a cash y `fill.size` (con signo) a position.

In [ ]:
class Fill:
    def __init__(self, side, price, size):
        self.side=side; self.price=price; self.size=size
    def cash_flow(self):
        return (-1 if self.side=='buy' else 1)*self.price*self.size
class PositionTracker:
    def __init__(self):
        pass
    def apply_fill(self, fill):
        pass

In [ ]:
t = PositionTracker()
t.apply_fill(Fill('buy',100,0.5))
assert abs(t._cash + 50) < 1e-9 and abs(t._position - 0.5) < 1e-9
print('ok')

### Solución guiada

```python
class PositionTracker:
    def __init__(self):
        self._cash = 0.0
        self._position = 0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow()
        self._position += fill.size if fill.side=='buy' else -fill.size
```

## 5. Equity a mercado

**Practicas:** componer estado.

Añade `equity(mark_price)` = cash + position * mark_price.

In [ ]:
class Fill:
    def __init__(self, side, price, size):
        self.side=side; self.price=price; self.size=size
    def cash_flow(self):
        return (-1 if self.side=='buy' else 1)*self.price*self.size
class PositionTracker:
    def __init__(self):
        self._cash=0.0; self._position=0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow()
        self._position += fill.size if fill.side=='buy' else -fill.size
    def equity(self, mark_price):
        pass

In [ ]:
t = PositionTracker()
t.apply_fill(Fill('buy',100,1))
assert abs(t.equity(110) - 10) < 1e-9, 'compra a 100, marca a 110 -> equity 10'
print('ok')

### Solución guiada

```python
# dentro de PositionTracker:
    def equity(self, mark_price):
        return self._cash + self._position * mark_price
```

## Cierre

Composición: un OrderBook contiene niveles; un PositionTracker consume Fills. Los objetos se hablan entre sí.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.